In [ ]:
import pandas as pd
import os

In [ ]:
import statistics as s
from math import isnan
from itertools import filterfalse
import numpy as np

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

In [ ]:
#getting csv with the viral concepts that will be chorts
concepts_csv = 'updated_concept_query_list_greater_than_100.csv'
viral_concept_df = pd.read_csv(concepts_csv)

#getting csv with the viral concepts that will be chorts
concepts_csv = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/viral_cohort_patient_overlap_seasonal_nonseasonal_update1.csv'
viral_concept_df2 = pd.read_csv(concepts_csv)

In [ ]:
viral_concept_df

In [ ]:
viral_concept_df2


In [ ]:
#find the incorrect counts and add those to a list and work only on those to correct and then run full code 
#and compare sets of the patients counts to do a final check before moving on. 

#make sure all duplicates are dropped from ALL tables prior to merging 
#there must be a logical error in the order of my wrangling that is altering my count once called inside the merge function

In [ ]:
def demographics_table():
    """
    Fetch person demographics rows for a single concept_id,
    using your original SQL structure and injecting concept_id directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    demographics_sql  = f"""
    SELECT
        person.person_id,
        
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
   
        p_race_concept.concept_name as race,
    
        p_ethnicity_concept.concept_name as ethnicity,
    
        p_sex_at_birth_concept.concept_name as sex_at_birth,
      
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `{dataset}.person` person 
    LEFT JOIN
        `{dataset}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{dataset}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_array_data = 1 ) )"""


    demographics_df = pd.read_gbq(
        demographics_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return demographics_df


In [ ]:
def socioeconomics_table():
    """
    Fetch socioeconomic observations mapped to ZIP-3 SES values,
    using your original SQL structure and injecting the dataset directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,

        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
      
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_array_data = 1 ) ) 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""


    socioeconomic_df = pd.read_gbq(
        socioeconomic_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return socioeconomic_df

In [ ]:
socio = socioeconomics_table().drop_duplicates('person_id')
socio.to_csv("/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/socioeconomics_table.csv")

In [ ]:
def get_condition_summary(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
      - first and last diagnosis date for that concept
      - condition_type_concept_name, visit_occurrence_concept_name from first occurrence
      - visits_for_concept: count of unique visit_occurrence_id for the concept
      - visits_all_concepts: count of unique visits across all conditions
      - concept_count_ehr: count of distinct condition concepts in EHR
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
      fd.standard_vocabulary,
      fd.first_diag_date,
      sm.last_diag_date,
      fd.condition_type_concept_name,
      fd.visit_occurrence_concept_name,
      sm.visits_for_concept,
      av.visits_all_concepts,
      cc.concept_count_ehr
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
def merge_data_table(): 
    
    cohort_dict = {}
    
    #read in concepts.csv and grab needed variables
    viral_concept_df = pd.read_csv(concepts_csv)
    concept_list = viral_concept_df['condition_concept_id'].tolist()
    concept_name = viral_concept_df['standard_concept_name'].to_list()
    
    
    # call demo and socio function for table and drop duplicates by person_id: one-row-per-patient tables once
    demo = demographics_table().drop_duplicates('person_id')
    socio = socioeconomics_table().drop_duplicates('person_id')

    for concept_id, concept in zip(concept_list, concept_name):
    
        cond = get_condition_summary(concept_id)
        
        merged = (
            cond
              .merge(demo,  on='person_id', how='left')
              .merge(socio, on='person_id', how='left')
        )
        
        
        cohort_dict[(concept_id, concept)] = merged
       

    return cohort_dict

In [ ]:
#use this when you only want to test a subset of cohorts
def merge_data_table(): 
    
    cohort_dict = {}
    
    #read in concepts.csv and grab needed variables
    viral_concept_df = pd.read_csv(concepts_csv)
    #concept_list = viral_concept_df['condition_concept_id'].tolist()
    concept_list = [37311061, 140641, 46273463,440329,197494,198964,435463,440021,260427,439727,74855,4241530,
                    137785,198075,441788,4092538,3661408,380038] 
    concept_name = viral_concept_df['standard_concept_name'].to_list()
    
    
    # call demo and socio function for table and drop duplicates by person_id: one-row-per-patient tables once
    demo = demographics_table().drop_duplicates('person_id')
    socio = socioeconomics_table().drop_duplicates('person_id')

    for concept_id, concept in zip(concept_list, concept_name):
    
        cond = get_condition_summary(concept_id)
        
        merged = (
            cond
              .merge(demo,  on='person_id', how='left')
              .merge(socio, on='person_id', how='left')
        )
        
        
        cohort_dict[(concept_id, concept)] = merged
       

    return cohort_dict

In [ ]:
cohort_data_tables = merge_data_table()

In [ ]:
cohort_data_tables

In [ ]:
def merge_race_ethnicity_data(new_column):
    
    if new_column["ethnicity"] == "Hispanic or Latino":
        return new_column["ethnicity"]
    else:
        return new_column["race"]
    
#apply function
cohort_data_dict = {}

for key, table in cohort_data_tables.items():
    table["updated_race"] = table.apply(merge_race_ethnicity_data, axis=1)
    
    cohort_data_dict[key] = table    
    

In [ ]:
cohort_data_dict

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/cohort_stats_df_for_all_concepts_cache.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(cohort_data_dict, f)

print(f"Saved {len(cohort_data_dict)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/cohort_stats_df_for_all_concepts_cache.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    cohort_data_dict = pickle.load(f)

print("Reloaded keys:", list(cohort_data_dict.keys()))

In [ ]:
#Checking that pt query and count match the results seen if using the cohort builder

In [ ]:
concept_id = list(viral_concept_df['condition_concept_id'])
len(concept_id)

In [ ]:
#checking by len('person_id') and set(person_id) matching


personIDs_len = []
personIDs = set()

for key, value in cohort_data_dict.items():
    count = len(value['person_id'])
    personIDs_len.append(count)
    
    person_ID = value['person_id']

    personIDs.update(person_ID)


In [ ]:
personIDs_len

In [ ]:
personIDs

In [ ]:
len(personIDs)

In [ ]:
# This query represents dataset "viral_overlap" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_76153600_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (440029)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_array_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (440029)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
                )
            ) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id"""

dataset_76153600_condition_df = pd.read_gbq(
    dataset_76153600_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

condition_df = dataset_76153600_condition_df

In [ ]:
condition_df['person_id'].nunique() #matches the cohort builder count for person_id under 440029

In [ ]:
#filering for concept_Id used as a cohort
condition_df_filtered = condition_df[condition_df['condition_concept_id'].isin(concept_id)].copy()
condition_df_filtered

In [ ]:
condition_df_filtered['person_id'].nunique() # count once noncohort concept_id dropped from code that comes from the cohort builder

In [ ]:
persons = condition_df_filtered['person_id']
cohort_builder_personIDs = set(persons)


In [ ]:
personIDs.difference(cohort_builder_personIDs)

In [ ]:
cohort_builder_personIDs.difference(personIDs)

In [ ]:
# SUMMARY STATS

In [ ]:
def stats_summary_dict(df_dict):
   
    '''This function is to create a dataframe of the stats necessary for further descriptive statitics) '''

    stats_dict = {}

    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple

        #adding age based on date_of_birth column
        table['age'] = table['first_diag_date'].dt.year - table['date_of_birth'].dt.year
        
        
        # Strip NONE and replace with NaN values
        table.replace("NONE", np.nan, inplace=True)
        
    
        #concept_name = no_dups['standard_concept_name']
        #sex = table['sex_at_birth']
        #socio = table['zip_code']
        #race_stats = table['race']
        #self_reported = table['self_reported_category']
        #ethnicity = table['updated_race']
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept']
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        cohort_count = list(table['person_id'])
        
        # Strip NaN values
        age = list(filterfalse(isnan, age_stats))
        concept_visit = list(filterfalse(isnan, concept_visit_stats))   
        gen_diag = list(filterfalse(isnan, gen_diag_stats))
        gen_visits = list(filterfalse(isnan, gen_visit_stats))
       
        
        #build dataframe of results
        df = pd.DataFrame ({
             
            'concept_id' : [cohort],
            'cohort count' : [len(cohort_count)],
            'age_median' : [s.median(age)],
            'age_mean' : [round(s.mean(age))],
            'age_min' : [min(age)],
            'age_max' : [max(age)],            
            'age_Q1' : [np.percentile(age, [25])],
            'age_Q3' : [np.percentile(age, [75])],
            
            
            'concept_visit_median' : [s.median(concept_visit)],
            'concept_visit_mean' : [s.mean(concept_visit)],
            'concept_visit_min' : [min(concept_visit)],
            'concept_visit_max' : [max(concept_visit)],            
            'concept_visit_Q1' : [np.percentile(concept_visit, [25])],
            'concept_visit_Q3' : [np.percentile(concept_visit, [75])],
            
            
            'total_diag_median' : [s.median(gen_diag)],
            'total_diag_mean' : [s.mean(gen_diag)],
            'total_diag_min' : [min(gen_diag)],
            'total_diag_max' : [max(gen_diag)],            
            'total_diag_Q1' : [np.percentile(gen_diag, [25])],
            'total_diag_Q3' : [np.percentile(gen_diag, [75])],
            
            
            'total_visits_median' : [s.median(gen_visits)],
            'total_visits_mean' : [s.mean(gen_visits)],
            'total_visits_min' : [min(gen_visits)],
            'total_visits_max' : [max(gen_visits)],            
            'total_visits_Q1' : [np.percentile(gen_visits, [25])],
            'total_visits_Q3' : [np.percentile(gen_visits, [75])]
     
        })
    
        #wrap results in a dictionary
        stats_dict[cohort] = df
     
    return stats_dict

In [ ]:
stats = stats_summary_dict(cohort_data_dict)
stats  #returns dict of of dataframes for each concept_id key

In [ ]:
stats_table = pd.concat(stats.values(), axis = 0, ignore_index = True)
stats_table
        

In [ ]:
stats_table.to_csv("viral_cohort_summary_stats.csv")

In [ ]:
import plotly.graph_objects as go

def make_table_figure(df, title):
    
    # transpose & reset index
    df_t = df.T.reset_index()
    df_t.columns = ["Descriptive stat", "Value"]

    # build the table figure
    fig = go.Figure(
        go.Table(
            header=dict(
                values=df_t.columns,
                height = 30,
                fill_color="black",
                font=dict(color="white", size=12),
                align="left"
            ),
            cells=dict(
                values=[df_t["Descriptive stat"], df_t["Value"]],
                height = 25,
                fill_color=["lightblue", "lightgray"],
                align="left"
            )
        )
    )
    fig.update_layout(
        title= "Viral disease cohort descriptive statatistics",
        height=300,
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig



fig_dict = {}

#loop through stats table results and apply the function to build the table
for name, df in stats.items():
    fig_dict[name] = make_table_figure(df, title=name)

In [ ]:
import os, plotly.io as pio

def make_plotly_table_html(table_dict):

    # 1) Make sure results/ exists
    output_dir = 'cohort_summary_table' #folder
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    df = table_dict
    

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral disease cohort descriptive stats table</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    
    
    output_path = "cohort_summary_table/cohorts_summary_plotly_tables.html" #output path, folder to save and html file name
    with open(output_path, "w", encoding="utf-8") as f:
        
        f.write(html_header) #writing the header 

        first = True  #Sets a flag to track whether this is the first figure being written. It ensures that Plotly JS is only included once.
        
        for cohort, figure in df.items():
            f.write(f"<h2>Cohort: {cohort}</h2>\n") #write the image header, notated in html_header (ie) like metadata)
          
            #convert plotly to an html snippet
            snippet = pio.to_html(
                figure,
                full_html=False,
                include_plotlyjs="inline" if first else False  # Ensures the Plotly JavaScript is only embedded once (for the first figure) to avoid bloating the file.
            )
            
            first = False #After writing the first figure, first is set to False so that Plotly JS isn’t redundantly included for subsequent figures.

  
            f.write(snippet + "<hr>\n") #adding plot

        f.write(html_footer) #adding footer

    print(f"✅ Wrote {output_path}") #completion
    
    

In [ ]:
make_plotly_table_html(fig_dict)

In [ ]:
def stats_summary_plots(df_dict):
   
    '''This function is to create distribution plots from the data from the merge_data_table() function'''

    figs_dict = {}

    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple
        
            
        #create series of data    
        sex = table['sex_at_birth']
        ethnicity = table['updated_race']
        socio = table['zip_code']
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept'] 
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        
        
        
        #create plots
        fig = make_subplots(rows = 2, cols = 4, vertical_spacing= 0.30, horizontal_spacing = .1)
          
        trace1 = go.Histogram(x = sex, name = 'sex')
        trace2 = go.Histogram(x = age_stats, name = 'age', xbins = dict(size = '1'), autobinx = False)
        trace3 = go.Histogram(x = ethnicity, name = 'ethnicity') 
        trace4 = go.Histogram(x = socio, name = 'zip code', xbins = dict(size = '1'), autobinx = False)
        trace5 = go.Histogram(x = concept_visit_stats, name = 'concept visit count', xbins = dict(size = '1'), autobinx = False)
        trace6 = go.Histogram(x = gen_diag_stats, name = 'total diagnosis count', xbins = dict(size = '1'), autobinx = False)
        trace7 = go.Histogram(x = gen_visit_stats, name = 'total visits', xbins = dict(size = '1'), autobinx = False)
        

            
        fig.add_trace(trace1, 1, 1)
        fig.add_trace(trace2, 1, 2)
        fig.add_trace(trace3, 1, 3)
        fig.add_trace(trace4, 1, 4)
        
        fig.add_trace(trace5, 2, 1)
        fig.add_trace(trace6, 2, 2)
        fig.add_trace(trace7, 2, 3)
  
       
        
        
        #update subplot axes
        #row 1
        fig.update_xaxes(title_text="sex", tickangle= -45, row = 1, col = 1)
        fig.update_yaxes(title_text="patient count", row = 1, col = 1)
        
        fig.update_xaxes(title_text="age", row = 1, col = 2)
        fig.update_yaxes(title_text="patient count", row = 1, col = 2)
        
        fig.update_xaxes(title_text='ethnicity', tickangle= -45, row = 1, col = 3)
        fig.update_yaxes(title_text="patient count", row = 1, col = 3)
        
        
        fig.update_xaxes(title_text="zip code", tickangle= -45, row = 1, col = 4)
        fig.update_yaxes(title_text="patient count", row = 1, col = 4)
        
        
        
        #row2
        fig.update_xaxes(title_text="unique concept visit count", row = 2, col = 1)
        fig.update_yaxes(title_text="patient count", row = 2, col = 1)
        
        fig.update_xaxes(title_text="diagnosis count", row = 2, col = 2)
        fig.update_yaxes(title_text="patient count", row = 2, col = 2)
        
        fig.update_xaxes(title_text="visits count", row = 2, col = 3)
        fig.update_yaxes(title_text="patient count", row = 2, col = 3)
        
        
        
        fig.update_xaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
        fig.update_yaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )

        
            
        fig.update_layout(title_text= F'Patient count distribution: {cohort}',legend_title_text = "Variable", 
                            plot_bgcolor='white', bargap = 0.2, height = 700, width = 1000, margin=dict(l=80, r=80, t=80, b=80))          

        figs_dict[cohort] = fig
        
    return figs_dict

In [ ]:
import os, plotly.io as pio

def create_cohort_summary_plots_html():

    # 1) Make sure results/ exists
    os.makedirs("results", exist_ok=True)

    # 2) Generate your dict of Figures
    figs_dict = stats_summary_plots(cohort_data_dict)

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral Disease Cohort Summary Statistics</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    output_path = "results/cohort_summary_plots.html"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_header)

        first = True
        for cohort_id, fig in figs_dict.items():
            f.write(f"<h2>Cohort: {cohort_id}</h2>\n")

            snippet = pio.to_html(
                fig,
                full_html=False,
                include_plotlyjs="inline" if first else False
            )
            first = False

            f.write(snippet + "\n<hr>\n")

        f.write(html_footer)

    print(f"✅ Wrote {output_path}")

In [ ]:
create_cohort_summary_plots_html()

In [ ]:
def diagnosis_date_freq(diagnosis_dates_df):
    
    """
    For each cohort in diagnosis_dates_df, plots histograms of
    first & last diagnosis dates, and saves each figure to a dictionary, with its respective concept_id key
    
    """
    
  
    date_dict = {}
    
    for cohort, table in diagnosis_dates_df.items():
            
              
            first = table['first_diag_date']
            last = table['last_diag_date']
    
            
            fig = make_subplots(rows = 1, cols = 2)
          

            
            first_trace = go.Histogram(x = first, name = 'first diagnosis', xbins = dict(
                                        size='M1'),autobinx = False)
            
            last_trace = go.Histogram(x = last, name = 'last diagnosis', xbins = dict(
                                     size='M1'), autobinx = False)
            
            
            fig.add_trace(first_trace, 1, 1)
            fig.add_trace(last_trace, 1, 2)
            
            
            
            fig.update_xaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
            fig.update_yaxes(automargin=True, title_font = dict(size = 12), tickfont = dict(size = 10), title_standoff = 10, showline=True, linecolor='black', linewidth= .05, mirror= False )
            
            fig.update_layout(title_text= F'First and Last diagnosis distribution: {cohort}', 
                xaxis_title_text='Diagnosis Date', 
                yaxis_title_text='Patient Count', 
                bargap = 0.2)
            
            
            date_dict[cohort] = fig
            
    return date_dict

In [ ]:
import os, plotly.io as pio

def create_cohort_date_plots_html():

    # 1) Make sure results/ exists
    output_dir = 'diagnosis_dates_plots'
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    dates_dict = diagnosis_date_freq(cohort_data_tables)


    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral Disease Cohort Date distributions</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    output_path = "diagnosis_dates_plots/all_cohorts_dates_plot.html"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_header)

        first = True
        for cohort_id, fig in dates_dict.items():
            f.write(f"<h2>Cohort: {cohort_id}</h2>\n")

            snippet = pio.to_html(
                fig,
                full_html=False,
                include_plotlyjs="inline" if first else False
            )
            first = False

            f.write(snippet + "\n<hr>\n")

        f.write(html_footer)

    print(f"✅ Wrote {output_path}")


In [ ]:
create_cohort_date_plots_html()

In [ ]:
#post cohort summary stats analysis

'''
1. identify outliers view skew test
    age, concept visit count, total visit count, and total diagnosis count.
    
2. Sex : identify any 70/30 proportion threshold within the chorts for sex bias
3. Race, zip code , find highest peak, and find what proportion of the cohort is found there 

input: df1 - original cohort dataframe dictionary

1; array of values from df1 table, similar to code used for the stats summary table
2: count of either men or women, and calculate the proportion to see if any meet the 70/30 threshhold
3. sum the count of each unique x value, take the highest count and calculate the proportion, series of these resuslts 

output: go.table with results
'''

In [ ]:
import scipy.stats as scipy
def post_cohort_analysis(df_dict): 
    
   
    post_stats_dict = {}

    for cohort, table in df_dict.items(): #need access to key also, to store results later in the loop / tuple


        
        # Strip NONE and replace with NaN values
        table.replace("NONE", np.nan, inplace=True)
        
    
        #concept_name = no_dups['standard_concept_name']
        sex = (table['sex_at_birth']).astype('category')
        socio = table['zip_code'].astype('category')
        ethnicity = table['updated_race'].astype('category')
        age_stats = table['age']
        concept_visit_stats = table['visits_for_concept']
        gen_diag_stats = table['concept_count_ehr']
        gen_visit_stats = table['visits_all_concepts']
        cohort_count = list(table['person_id'])
        
        # Strip NaN values
        age = list(filterfalse(isnan, age_stats))
        concept_visit = list(filterfalse(isnan, concept_visit_stats))   
        gen_diag = list(filterfalse(isnan, gen_diag_stats))
        gen_visits = list(filterfalse(isnan, gen_visit_stats))
       
        
        
        # 1. identify outliers via skew test
    
        age_skew = round(scipy.skew(age), 3)
        concept_visit_skew = round(scipy.skew(concept_visit), 3)
        total_diag_skew = round(scipy.skew(gen_diag), 3)
        total_visit_skew = round(scipy.skew(gen_visits), 3)
        
        
        
        
        # 2. Sex : identify any 70/30 proportion threshold within the chorts for sex bias
        #use female as proxy
     
        male = round(sex.isin(['Male']).sum() / len(cohort_count) * 100, 2)
        female = round(sex.isin(['Female']).sum() / len(cohort_count) * 100, 2)
 
        
        
        # 3. Race, zip code , find highest peak, and find what proportion of the cohort is found there 

        max_ethnicity = ethnicity.value_counts().max()  #max race value
        which_ethnicity = ethnicity.value_counts().idxmax()
     
        max_zip = socio.value_counts().max()
        which_zip = socio.value_counts().idxmax()
        
        max_ethnicity_prop = round(max_ethnicity / len(cohort_count) * 100, 2)
        max_zip_prop = round(max_zip / len(cohort_count) * 100, 2)
        
       
               
        #build dataframe of results
        df = pd.DataFrame ({ 
            
            'concept_id' : [cohort], 
            'male_prop' : [male],
            'female_prop' : [female],
            'max_ethnicity' : [which_ethnicity],
            'max_ethnicity_prop' : [max_ethnicity_prop],
            'max_zip' : [which_zip],
            'max_zip_prop' : [max_zip_prop],
            'skew_of_age' : [age_skew],
            'skew_of_concept_visits' : [concept_visit_skew],
            'skew_of_total_diag' : [total_diag_skew],            
            'skew_of_total_visits' : [total_visit_skew],

               })
        
           #wrap results in a dictionary
        post_stats_dict[cohort] = df
     
    return post_stats_dict

In [ ]:
post_analysis_dict = post_cohort_analysis(cohort_data_tables)
post_analysis_dict

In [ ]:
post_summary_dict = {}


#loop through stats table results and apply the function to build the table
for name, df in post_analysis_dict .items():
    post_summary_dict[name] = make_table_figure(df, title=name) #used previous function created for the stats summary
    
    
post_summary_dict

In [ ]:
import os, plotly.io as pio

def post_summary_plotly_table_html(post_table_dict):

    # 1) Make sure results/ exists
    output_dir = 'post_cohort_summary_table' #folder
    os.makedirs(output_dir, exist_ok=True)

    # 2) Generate your dict of Figures
    df = post_table_dict
    

    # 3) HTML header (no <script> tag here)
    html_header = """<!DOCTYPE html>
    <html><head>
      <meta charset="utf-8">
      <title>Cohort Gallery</title>
      <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        h1   { text-align: center; margin-bottom: 40px; }
        h2   { margin-top: 50px; }
        hr   { border: none; border-top: 1px solid #ddd; margin: 30px 0; }
      </style>
    </head><body>
      <h1>Viral disease cohort post descriptive stats table</h1>
    """

    html_footer = "</body></html>"

    # 4) Write out the page, inlining Plotly.js the first time
    
    
    output_path = "post_cohort_summary_table/post_cohorts_summary_plotly_tables.html" #output path, folder to save and html file name
    with open(output_path, "w", encoding="utf-8") as f:
        
        f.write(html_header) #writing the header 

        first = True  #Sets a flag to track whether this is the first figure being written. It ensures that Plotly JS is only included once.
        
        for cohort, figure in df.items():
            f.write(f"<h2>Cohort: {cohort}</h2>\n") #write the image header, notated in html_header (ie) like metadata)
          
            #convert plotly to an html snippet
            snippet = pio.to_html(
                figure,
                full_html=False,
                include_plotlyjs="inline" if first else False  # Ensures the Plotly JavaScript is only embedded once (for the first figure) to avoid bloating the file.
            )
            
            first = False #After writing the first figure, first is set to False so that Plotly JS isn’t redundantly included for subsequent figures.

  
            f.write(snippet + "<hr>\n") #adding plot

        f.write(html_footer) #adding footer

    print(f"✅ Wrote {output_path}") #completion

In [ ]:
post_summary_plotly_table_html(post_summary_dict)
